# DEH 30-Day PySpark Challenge
## Day 3 — Schema & DataFrame Reader

**Instructions:**
1. Click `File → Save a copy in Drive` before editing
2. Run each cell in order using `Shift + Enter`
3. Complete the assignment cells at the bottom

---

In [ ]:
!pip install pyspark --quiet

In [ ]:
import os
os.environ['AWS_ACCESS_KEY_ID']     = 'your-access-key-here'
os.environ['AWS_SECRET_ACCESS_KEY'] = 'your-secret-key-here'
os.environ['AWS_DEFAULT_REGION']    = 'us-east-1'
BUCKET = 'deh-pyspark-challenge-your-account-id'
print('Credentials set.')

In [ ]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField,
    StringType, IntegerType, DoubleType, DateType, LongType, BooleanType, TimestampType
)

spark = (
    SparkSession.builder
    .appName("DEH-PySpark-Challenge")
    .config("spark.jars.packages",
            "org.apache.hadoop:hadoop-aws:3.4.0,com.amazonaws:aws-java-sdk-bundle:1.12.367")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.aws.credentials.provider",
            "com.amazonaws.auth.EnvironmentVariableCredentialsProvider")
    .getOrCreate()
)
print(f"Spark version: {spark.version}")

## Part 1 — Schema

### Step 4 — Default read (no schema)

In [ ]:
# Default read — everything is StringType
orders_default = spark.read.csv(f"s3a://{BUCKET}/data/orders.csv", header=True)
print("--- Default schema ---")
orders_default.printSchema()

### Step 5 — Explicit schema with StructType

In [ ]:
orders_schema = StructType([
    StructField("order_id",       StringType(),   True),
    StructField("customer_id",    StringType(),   True),
    StructField("product_id",     StringType(),   True),
    StructField("order_date",     DateType(),     True),
    StructField("quantity",       IntegerType(),  True),
    StructField("unit_price",     DoubleType(),   True),
    StructField("discount_pct",   IntegerType(),  True),
    StructField("status",         StringType(),   True),
    StructField("payment_method", StringType(),   True),
    StructField("region",         StringType(),   True)
])

orders_df = spark.read.csv(
    f"s3a://{BUCKET}/data/orders.csv",
    header=True,
    schema=orders_schema
)

print("--- Explicit schema ---")
orders_df.printSchema()
orders_df.show(3)

## Part 2 — DataFrame Reader API

### Step 6 — Two styles of passing options

In [ ]:
# Style 1 — keyword arguments
df1 = spark.read.csv(f"s3a://{BUCKET}/data/orders.csv", header=True, inferSchema=True)

# Style 2 — chained .option() calls (preferred in production)
df2 = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv(f"s3a://{BUCKET}/data/orders.csv")

print(f"Both styles same result: {df1.count() == df2.count()}")

### Step 7 — Reading with multiple options and explicit schema

In [ ]:
orders_df = spark.read \
    .option("header", "true") \
    .option("dateFormat", "yyyy-MM-dd") \
    .option("nullValue", "N/A") \
    .option("mode", "PERMISSIVE") \
    .schema(orders_schema) \
    .csv(f"s3a://{BUCKET}/data/orders.csv")

orders_df.printSchema()
orders_df.show(3)

## Part 3 — Reading Modes

### Step 8 — PERMISSIVE with _corrupt_record (on dirty data)

In [ ]:
schema_with_corrupt = StructType([
    StructField("order_id",        StringType(),  True),
    StructField("customer_id",     StringType(),  True),
    StructField("product_id",      StringType(),  True),
    StructField("order_date",      DateType(),    True),
    StructField("quantity",        IntegerType(), True),
    StructField("unit_price",      DoubleType(),  True),
    StructField("discount_pct",    IntegerType(), True),
    StructField("status",          StringType(),  True),
    StructField("payment_method",  StringType(),  True),
    StructField("region",          StringType(),  True),
    StructField("_corrupt_record", StringType(),  True)
])

df_permissive = spark.read \
    .option("header", "true") \
    .option("mode", "PERMISSIVE") \
    .option("columnNameOfCorruptRecord", "_corrupt_record") \
    .schema(schema_with_corrupt) \
    .csv(f"s3a://{BUCKET}/data/orders_dirty.csv")

# cache() required before filtering on _corrupt_record
df_permissive.cache()

total = df_permissive.count()
bad   = df_permissive.filter(df_permissive._corrupt_record.isNotNull()).count()
print(f"Total rows : {total}")
print(f"Bad rows   : {bad}")
df_permissive.filter(df_permissive._corrupt_record.isNotNull()) \
    .select("_corrupt_record").show(truncate=False)

### Step 9 — DROPMALFORMED

In [ ]:
df_drop = spark.read \
    .option("header", "true") \
    .option("mode", "DROPMALFORMED") \
    .schema(orders_schema) \
    .csv(f"s3a://{BUCKET}/data/orders_dirty.csv")

# cache() needed — count() without cache gives inconsistent results
df_drop.cache()
df_drop.count()  # trigger the cache

print(f"Rows with DROPMALFORMED: {df_drop.count()}")
# Result: 12 — 3 rows with wrong column count dropped
# Type mismatches are nulled out and KEPT — not dropped

### Step 10 — FAILFAST

In [ ]:
# Use show() not count() — show() scans rows physically and triggers the exception
df_strict = spark.read \
    .option("header", "true") \
    .option("mode", "FAILFAST") \
    .schema(orders_schema) \
    .csv(f"s3a://{BUCKET}/data/orders_dirty.csv")

df_strict.show()
# Throws SparkException immediately on the first bad row

---
## Assignment — Day 3

---

### Task 1
Read `products.csv` from S3 with no options. Print the schema.  
Then read it again with `inferSchema=True`. What columns changed type?

In [ ]:
# Task 1 — Write your code here


### Task 2
Define an explicit `StructType` schema for `products.csv`:  
`product_id` (String), `product_name` (String), `category` (String), `sub_category` (String), `unit_price` (Double), `cost_price` (Double), `supplier` (String), `stock_quantity` (Integer).  
Read the file and verify the schema.

In [ ]:
# Task 2 — Write your code here


### Task 3
Read `customers.csv` from S3 using the chained `.option()` style with an explicit schema.  
Columns: `customer_id`, `first_name`, `last_name`, `email`, `city`, `state`, `country`, `signup_date` (Date), `segment`.  
Show the first 5 rows.

In [ ]:
# Task 3 — Write your code here


### Task 4
Read `orders_dirty.csv` using PERMISSIVE mode with `_corrupt_record`.  
Print total rows and bad row count.  
Then read it with DROPMALFORMED — compare the counts.  
Why does DROPMALFORMED still return 15 rows for type mismatches but drops the 3 column-count rows?

In [ ]:
# Task 4 — Write your code here


---
*DEH 30-Day PySpark Challenge · Data Engineering Hub · RADE Program*